In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

All libraries imported successfully!
Pandas version: 2.2.2
Numpy version: 1.26.4


In [2]:
# Load the processed dataset exported from MySQL
df = pd.read_csv(r'C:\Users\AmalDev\Downloads\processed_battery_data.csv')

# Basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape: (1415, 9)

Columns: ['battery_id', 'cycle', 'voltage', 'temperature', 'capacity', 'soh', 'rul', 'health_label', 'degradation_pct']

Data types:
battery_id          object
cycle                int64
voltage            float64
temperature        float64
capacity           float64
soh                float64
rul                  int64
health_label        object
degradation_pct    float64
dtype: object

First 5 rows:


,battery_id,cycle,voltage,temperature,capacity,soh,rul,health_label,degradation_pct
0,B0005,1,3.53278,32.5369,1.86198,1.000000,167,Healthy,0.00
1,B0005,2,3.54297,32.6436,1.85186,0.994568,166,Healthy,0.54
2,B0005,3,3.55306,32.5225,1.84081,0.988631,165,Healthy,1.14
3,B0005,4,3.54585,32.4921,1.85006,0.993599,164,Healthy,0.64
4,B0005,5,3.54446,32.3686,1.84943,0.993263,163,Healthy,0.67


In [3]:
# Data quality check
print("=== DATA QUALITY REPORT ===")
print(f"\nTotal rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

print("\nNull values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nBasic statistics:")
df.describe()

=== DATA QUALITY REPORT ===

Total rows: 1415
Total columns: 9

Null values per column:
battery_id         0
cycle              0
voltage            0
temperature        0
capacity           0
soh                0
rul                0
health_label       0
degradation_pct    0
dtype: int64

Duplicate rows: 0

Basic statistics:


,cycle,voltage,temperature,capacity,soh,rul,degradation_pct
count,1415.000000,1415.000000,1415.000000,1415.000000,1415.000000,1415.000000,1415.000000
mean,55.389399,3.420612,26.589774,1.501191,0.849014,54.175972,15.098594
std,42.728497,0.166460,13.196046,0.316995,0.119219,43.709204,11.921819
min,1.000000,2.530010,7.684550,0.520105,0.604449,0.000000,-4.850000
25%,21.000000,3.387080,9.555185,1.321640,0.750777,16.500000,3.880000
50%,43.000000,3.452600,31.139300,1.504110,0.850350,43.000000,14.960000
75%,87.000000,3.507450,32.753400,1.709025,0.961174,86.000000,24.925000
max,168.000000,3.753030,54.997400,2.499000,1.048530,167.000000,39.560000


In [4]:
# Feature engineering — adding new columns
# 1. Voltage drop rate per battery
df['voltage_drop'] = df.groupby('battery_id')['voltage'].diff().fillna(0)

# 2. SOH lag features (previous cycle SOH)
df['soh_lag_1'] = df.groupby('battery_id')['soh'].shift(1).fillna(df['soh'])
df['soh_lag_3'] = df.groupby('battery_id')['soh'].shift(3).fillna(df['soh'])

# 3. Temperature x capacity interaction
df['temp_capacity'] = df['temperature'] * df['capacity']

# 4. Encode health_label as numbers
df['health_encoded'] = df['health_label'].map({
    'Healthy': 2,
    'Warning': 1, 
    'Critical': 0
})

print("New columns added successfully!")
print("Shape after feature engineering:", df.shape)
print("\nNew columns:")
print(df[['voltage_drop', 'soh_lag_1', 'soh_lag_3', 
          'temp_capacity', 'health_encoded']].head())

New columns added successfully!
Shape after feature engineering: (1415, 14)

New columns:
   voltage_drop  soh_lag_1  soh_lag_3  temp_capacity  health_encoded
0       0.00000   1.000000   1.000000      60.583057               2
1       0.01019   1.000000   0.994568      60.451377               2
2       0.01009   0.994568   0.988631      59.867743               2
3      -0.00721   0.988631   1.000000      60.112335               2
4      -0.00139   0.993599   0.994568      59.863460               2


In [5]:
# Save preprocessed data for use in modelling notebook
df.to_csv(r'C:\Users\AmalDev\Downloads\final_features.csv', index=False)
print("Saved successfully!")
print(f"Final dataset shape: {df.shape}")
print(f"\nAll columns: {df.columns.tolist()}")

Saved successfully!
Final dataset shape: (1415, 14)

All columns: ['battery_id', 'cycle', 'voltage', 'temperature', 'capacity', 'soh', 'rul', 'health_label', 'degradation_pct', 'voltage_drop', 'soh_lag_1', 'soh_lag_3', 'temp_capacity', 'health_encoded']
